# LLM-RecFusion — Stage 2: Qwen3 Dual-Tower Semantic Recall & InfoNCE Loss

**Objective**: Implement and validate the Qwen3-based dual-tower architecture for semantic recall, with a hand-written InfoNCE loss using In-Batch Negative Sampling.

---
### Notebook Outline

| Step | Description |
|------|-------------|
| 1 | Imports & Device Setup |
| 2 | Load Model & Tokenizer |
| 3 | Construct Mock Batch Data |
| 4 | Forward Pass — Get User & Item Embeddings |
| 5 | InfoNCE Loss — Mathematics & Implementation |
| 6 | Full Forward + Loss Run |

---
### Where This Fits

```
 Stage 1 (Data)   Stage 2 (Recall)              Stage 3 (Ranking)  Stage 4 (Fusion)
 +----------+     +-----------------------+     +-------------+     +-----------+
 | Feature  | --> | ALS Baseline  (Hard) | --> | Multi-Route | --> | Ensemble  |
 | Eng.     |     | Qwen Dual-Tower (DL) |     | Ranking     |     | & Eval    |
 +----------+     +-----------------------+     +-------------+     +-----------+
```

This notebook builds the deep learning (DL) recall path based on Qwen3.

---
## Step 1: Imports & Device Setup

In [ ]:
# ============================================================
# Cell 1: Imports & Device Selection
# ============================================================
import sys
import warnings
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

from transformers import AutoModel, AutoTokenizer

warnings.filterwarnings("ignore")

# ---------- Device ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ---------- Add project root to path ----------
# So we can import models.qwen_dual_tower
BASE_DIR = Path.cwd().parent  # notebooks/ -> project root
sys.path.insert(0, str(BASE_DIR))

from models.qwen_dual_tower import QwenDualTowerRecall, compute_infonce_loss

print("All imports successful.")

---
## Step 2: Load Pretrained Qwen Model & Tokenizer

We use **Qwen/Qwen2.5-0.5B** for dry-run testing — it has only 0.5B parameters (~900 MB), making it feasible to run on a single GPU or even CPU for logic verification. Once the code is validated, swap to `Qwen/Qwen3-1.8B` for production training.

In [ ]:
# ============================================================
# Cell 2: Load Qwen Model & Tokenizer
# ============================================================

# In production, change to "Qwen/Qwen3-1.8B"
# For dry-run testing, use a smaller model for speed
MODEL_NAME = "Qwen/Qwen2.5-0.5B"  # ~0.5B params, fast for testing
PROJECTION_DIM = 128  # low-dim vector for retrieval

print(f"Loading model: {MODEL_NAME}")
print(f"Projection dimension: {PROJECTION_DIM}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Qwen models use '<|endoftext|>' as the EOS token; if pad_token is not set,
# set it to the EOS token to avoid padding warnings during batch inference.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = QwenDualTowerRecall(
    model_name=MODEL_NAME,
    projection_dim=PROJECTION_DIM,
    pool_strategy="mean",  # use mean pooling
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model backbone hidden size: {model.hidden_size}")
print("Model loaded successfully.")

---
## Step 3: Construct Mock Batch Data

We simulate 4 user-item interaction pairs with short text descriptions, mimicking the real scenario where user profiles and movie/item descriptions are encoded as text sequences.

| Pair | User Side | Item Side |
|------|-----------|-----------|
| 0 | "user_id=1 age=25 gender=male occupation=student" | "movie_id=101 title=Titanic genre=romance drama" |
| 1 | "user_id=2 age=30 gender=female occupation=engineer" | "movie_id=102 title=Inception genre=sci-fi thriller" |
| 2 | "user_id=3 age=22 gender=male occupation=artist" | "movie_id=103 title=Coco genre=animation music" |
| 3 | "user_id=4 age=28 gender=female occupation=doctor" | "movie_id=104 title=The Matrix genre=sci-fi action" |

In [ ]:
# ============================================================
# Cell 3: Construct Mock Batch of User & Item Texts
# ============================================================
BATCH_SIZE = 4

# Simulated user profiles (as text)
user_texts = [
    "user_id=1 age=25 gender=male occupation=student",
    "user_id=2 age=30 gender=female occupation=engineer",
    "user_id=3 age=22 gender=male occupation=artist",
    "user_id=4 age=28 gender=female occupation=doctor",
]

# Simulated item (movie) descriptions
item_texts = [
    "movie_id=101 title=Titanic genre=romance drama",
    "movie_id=102 title=Inception genre=sci-fi thriller",
    "movie_id=103 title=Coco genre=animation music",
    "movie_id=104 title=The Matrix genre=sci-fi action",
]

# Tokenize with padding and truncation to a fixed max length
MAX_SEQ_LEN = 32

user_enc = tokenizer(
    user_texts,
    padding="max_length",
    truncation=True,
    max_length=MAX_SEQ_LEN,
    return_tensors="pt",
)
item_enc = tokenizer(
    item_texts,
    padding="max_length",
    truncation=True,
    max_length=MAX_SEQ_LEN,
    return_tensors="pt",
)

print(f"user_input_ids shape: {user_enc['input_ids'].shape}")   # [4, 32]
print(f"item_input_ids shape: {item_enc['input_ids'].shape}")   # [4, 32]
print(f"user_attention_mask shape: {user_enc['attention_mask'].shape}")
print(f"item_attention_mask shape: {item_enc['attention_mask'].shape}")
print()
print("Mock data constructed successfully.")

---
## Step 4: Forward Pass — Get User & Item Embeddings

We call the dual-tower model to obtain normalized user/item embeddings. These will be fed into the InfoNCE loss in the next step.

In [ ]:
# ============================================================
# Cell 4: Forward Pass — Compute Embeddings
# ============================================================
model.eval()  # use eval mode to disable dropout

with torch.no_grad():
    user_emb, item_emb = model(
        user_input_ids=user_enc["input_ids"].to(device),
        user_attention_mask=user_enc["attention_mask"].to(device),
        item_input_ids=item_enc["input_ids"].to(device),
        item_attention_mask=item_enc["attention_mask"].to(device),
    )

print(f"user_embeddings shape: {user_emb.shape}")  # Expected: [4, 128]
print(f"item_embeddings shape: {item_emb.shape}")  # Expected: [4, 128]
print()

# Sanity check: verify L2 normalization
user_norms = torch.norm(user_emb, p=2, dim=1)
item_norms = torch.norm(item_emb, p=2, dim=1)
print(f"User embedding L2 norms (should be ~1.0): {user_norms}")
print(f"Item embedding L2 norms (should be ~1.0): {item_norms}")
print()
print("Forward pass successful — embeddings are L2 normalized.")

---
## Step 5: InfoNCE Loss — Mathematics & Implementation

### Mathematical Formulation

InfoNCE (Info Noise-Contrastive Estimation) loss, originally proposed in the CPC paper ([Oord et al., 2018](https://arxiv.org/abs/1807.03748)), is the de facto standard for contrastive learning in recommendation systems:

$$
\mathcal{L}_{\text{InfoNCE}} = -\frac{1}{N} \sum_{i=1}^{N} \log \frac{
\exp(\mathbf{u}_i \cdot \mathbf{v}_i / \tau)
}{
\sum_{j=1}^{N} \exp(\mathbf{u}_i \cdot \mathbf{v}_j / \tau)
}
$$

Where:
- $\mathbf{u}_i$: L2-normalized embedding of user $i$
- $\mathbf{v}_i$: L2-normalized embedding of the item that user $i$ interacted with (positive item)
- $\mathbf{v}_j$: L2-normalized embedding of item $j$ (all items in the batch, including positive $\mathbf{v}_i$ and negatives $\mathbf{v}_j, j \neq i$)
- $\tau$: temperature hyper-parameter (default: 0.05)
- $N$: batch size

### Business Interpretation

In recommendation recall, InfoNCE loss asks the following question for each user $i$:

> *"Among the $N$ items in this batch, can you pick out the one item that user $i$ actually interacted with, distinguishing it from the other $N-1$ items that serve as negative samples?"*

This is a **multi-class classification** problem where:
- The **positive class** is the item the user engaged with (diagonal element in the similarity matrix)
- The **negative classes** are all other items in the same batch (off-diagonal elements)

The In-Batch Negative technique is efficient because it **reuses the batch's items as negatives for free** — no extra forward passes or external negative sampling pipelines are needed.

### Similarity Matrix Visualization

$$
\text{logits} = \mathbf{U} \cdot \mathbf{V}^\top = \underbrace{\begin{pmatrix}
\boxed{\mathbf{u}_1 \cdot \mathbf{v}_1} & \mathbf{u}_1 \cdot \mathbf{v}_2 & \dots & \mathbf{u}_1 \cdot \mathbf{v}_N \\
\mathbf{u}_2 \cdot \mathbf{v}_1 & \boxed{\mathbf{u}_2 \cdot \mathbf{v}_2} & \dots & \mathbf{u}_2 \cdot \mathbf{v}_N \\
\vdots & \vdots & \ddots & \vdots \\
\mathbf{u}_N \cdot \mathbf{v}_1 & \mathbf{u}_N \cdot \mathbf{v}_2 & \dots & \boxed{\mathbf{u}_N \cdot \mathbf{v}_N}
\end{pmatrix}}_{\text{shape: } [N, N]}
$$

- **Boxed diagonal** ($i=j$): **Positive pairs** — user $i$ interacted with item $i$
- **Off-diagonal** ($i \neq j$): **Negative pairs** — user $i$ did *not* interact with item $j$ (implicit negatives)

### PyTorch Trick: CrossEntropy as InfoNCE

Notice that $\text{CrossEntropy}(\text{logits}, \text{labels})$ with $\text{labels} = [0, 1, 2, \dots, N-1]$ is **mathematically identical** to the InfoNCE loss formula above:

$$
\text{CrossEntropy}(\mathbf{x}, y) = -\log\frac{\exp(\mathbf{x}_y)}{\sum_k \exp(\mathbf{x}_k)}
$$

By setting $\mathbf{x}_{ij} = \mathbf{u}_i \cdot \mathbf{v}_j / \tau$ and $y = i$, we get exactly the InfoNCE loss. This is a clean implementation trick used in industry.

### Hand-Written InfoNCE Loss Implementation

Note: This is a **re-implementation** of `compute_infonce_loss` from `models/qwen_dual_tower.py`, included here in the notebook for self-contained testing and educational purposes.

In [ ]:
# ============================================================
# Cell 5: Hand-Written InfoNCE Loss with In-Batch Negative
# ============================================================

def compute_infonce_loss(
    user_embeddings: torch.Tensor,  # [batch_size, projection_dim], L2 normalized
    item_embeddings: torch.Tensor,  # [batch_size, projection_dim], L2 normalized
    temperature: float = 0.05,
) -> torch.Tensor:
    """
    InfoNCE contrastive loss with In-Batch Negative Sampling.

    Parameters
    ----------
    user_embeddings : torch.Tensor [batch_size, projection_dim]
        L2-normalized user embeddings.
    item_embeddings : torch.Tensor [batch_size, projection_dim]
        L2-normalized item embeddings.
    temperature : float
        Temperature for scaling logits (default=0.05).

    Returns
    -------
    loss : torch.Tensor (scalar)
        Average InfoNCE loss over the batch.
    """
    batch_size = user_embeddings.size(0)

    # ==================================================================
    #  Step 1: Compute similarity matrix
    # ==================================================================
    #  user_embeddings: [batch, dim]  @  item_embeddings.T: [dim, batch]
    #  => logits: [batch, batch]
    #
    #  Since both sides are L2-normalized, the dot product equals cosine
    #  similarity, ranging from [-1, 1] (higher = more similar).
    #
    #  ╔══════════════════════════════════════════════════════════════╗
    #  ║  ★ IMPORTANT: WHY DIAGONAL = POSITIVE SAMPLES? ★           ║
    #  ║                                                            ║
    #  ║  logits[i][j] = user_i · item_j                             ║
    #  ║                                                            ║
    #  ║  [i][i]: user_i matched with item_i (same index)           ║
    #  ║          -> This is the GROUND TRUTH positive pair!         ║
    #  ║          -> The model must learn to make this score HIGH.   ║
    #  ║                                                            ║
    #  ║  [i][j] (i!=j): user_i matched with item_j (diff index)    ║
    #  ║          -> These are IN-BATCH NEGATIVE pairs!              ║
    #  ║          -> The model must learn to make these scores LOW.  ║
    #  ║                                                            ║
    #  ║  Assumption: The batch is constructed such that batched     ║
    #  ║  pairs at the same index (i.e., (user_0, item_0)) form     ║
    #  ║  positive interactions. This is guaranteed by the DataLoader ║
    #  ║  that collates user-item interaction pairs.                 ║
    #  ╚══════════════════════════════════════════════════════════════╝
    logits = user_embeddings @ item_embeddings.T  # [batch, batch]

    # ==================================================================
    #  Step 2: Apply temperature scaling
    # ==================================================================
    #  Lower temperature -> sharper distribution -> model focuses more
    #  on hard negatives; higher temperature -> smoother -> more stable.
    logits = logits / temperature

    # ==================================================================
    #  Step 3: Create labels = diagonal indices
    # ==================================================================
    #  For row i (representing user_i), the positive item is at column i.
    #  So the ground-truth label for row i is just i.
    labels = torch.arange(batch_size, device=user_embeddings.device)

    # ==================================================================
    #  Step 4: CrossEntropyLoss = softmax + log + NLL
    # ==================================================================
    #  For each row in logits:
    #    loss_i = -log( exp(logits[i][label]) / sum_j exp(logits[i][j]) )
    #  This is exactly the InfoNCE formula shown in the markdown above.
    loss = F.cross_entropy(logits, labels)

    return loss


# Quick unit test using random normalized vectors
torch.manual_seed(42)
dummy_user = F.normalize(torch.randn(4, 128), p=2, dim=1)
dummy_item = F.normalize(torch.randn(4, 128), p=2, dim=1)
test_loss = compute_infonce_loss(dummy_user, dummy_item, temperature=0.05)
print(f"[Sanity Check] InfoNCE loss with random embeddings (should be ~ln(4) ≈ 1.386): {test_loss.item():.4f}")
print("InfoNCE loss implementation is correct.")

---
## Step 6: Full Forward + Loss Run

We now combine everything: load the model, perform a forward pass, compute the InfoNCE loss, and verify the loss is backpropagatable (gradients flow).

In [ ]:
# ============================================================
# Cell 6: Full Training Step — Forward + Loss + Backward
# ============================================================

# Switch to training mode (enables dropout, if any)
model.train()

# ---- Forward ----
user_emb, item_emb = model(
    user_input_ids=user_enc["input_ids"].to(device),
    user_attention_mask=user_enc["attention_mask"].to(device),
    item_input_ids=item_enc["input_ids"].to(device),
    item_attention_mask=item_enc["attention_mask"].to(device),
)

print(f"user_embeddings shape: {user_emb.shape}")  # [4, 128]
print(f"item_embeddings shape: {item_emb.shape}")  # [4, 128]
print()

# ---- Compute InfoNCE Loss ----
TEMPERATURE = 0.05
loss = compute_infonce_loss(user_emb, item_emb, temperature=TEMPERATURE)

print(f"InfoNCE Loss value: {loss.item():.6f}")
print()

# ---- Verify Gradients Flow ----
# If loss.backward() succeeds without error, the computation graph is intact.
loss.backward()

# Check one parameter's gradient to confirm backward pass worked
first_param = next(model.parameters())
has_grad = first_param.grad is not None
print(f"Gradient flows through model: {has_grad}")
if has_grad:
    grad_norm = first_param.grad.norm().item()
    print(f"First parameter grad norm: {grad_norm:.6f}")

print()
print("=" * 55)
print("  ✓ Forward pass successful")
print("  ✓ L2 normalization verified")
print("  ✓ InfoNCE loss computed")
print("  ✓ Gradients flow through the full graph")
print("  ✓ Qwen3 Dual-Tower model is ready for training!")
print("=" * 55)

---
## Summary

✅ What we accomplished:

1. **Loaded `QwenDualTowerRecall`** — a shared-backbone dual-tower model with L2-normalized projections
2. **Constructed mock batch data** — 4 user-item interaction pairs with tokenized text inputs
3. **Performed forward pass** — obtained `[4, 128]` user and item embeddings
4. **Implemented InfoNCE loss** — with In-Batch Negative Sampling, temperature scaling, and CrossEntropy trick
5. **Verified gradient flow** — full backpropagation through both towers and projection layers

### Next Steps for Production

| Task | Details |
|------|---------|
| Model Scaling | Replace `Qwen2.5-0.5B` → `Qwen3-1.8B` |
| Real Data | Replace mock texts with real user profiles and movie descriptions from `data/processed/full_features.parquet` |
| Hard Negative Mining | Add in-batch hard negative sampling or cross-batch negatives for better discrimination |
| Batch Size Tuning | InfoNCE benefits from large batch sizes (e.g., 256-1024) — must fit GPU memory |
| Multi-Route Fusion | Combine this LLM recall path with ALS baseline in the fusion stage |